# Kartu-Verb project


The Kartu-Verb database comprises data pertaining to inflected Georgian verbs and their associated characteristics. The data is stored in a CSV file, with information organized into the following fields:

* form: The inflected form of a Georgian verb.
* tense_in_paradigm: The tense of the inflected form.
* person: The person of the inflected form (1st, 2nd, 3rd).
* number: The number of the inflected form (singular, plural).
* preverb: The preverb associated with the inflected form.
* pre2: The preradical of the inflected form.
* root: The root of the inflected form.
* sf2: The stem formant of the inflected form.
* caus_sf: The causative stem formant of the inflected form.
* ending: The ending of the inflected form.
* tsch_class: the Tschkhenkeli class to which the form belongs.
* morph_type: the morphology type to which the form belongs.
* id: Id in Clarino database to keep link to the corresponding croot.
* sub_id: Id in Clarino database to keep link to the corresponding verb paradigm.
* vn: Verbal Noun for the inflected form.

The objective of the project is to develop a model that can predict missing Verbal Nouns based on the provided information, including the form, tense_in_paradigm, person, number, preverb, pre2, root, sf2, caus_sf, ending, tsch_class, morph_type, id and sub_id.

# Import Libraries

Import the usual libraries for pandas and plotting. We can import sklearn later on.

In [1]:
import pandas as pd
import numpy as np
import re
%config InlineBackend.figure_formats = ['svg'] #pdf,svg
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime

## Get the Data

Read the Kartu-verb .csv file and assign it to a data frame named "kv".

In [2]:
kv = pd.read_csv('data_vn+withnotfullcroots-lat', sep=';')

Check the head of kv.

In [3]:
kv.head(5)

,form,tense_in_paradigm,person,number,preverb,pre2,root,sf2,caus_sf,ending,tsch_class,morph_type,sub_id,id,vn
0,vamxanagob,present,1,sg,-,v,amxanag,ob,-,-,MV,active,39-1,39,*amxanagoba
1,hamxanagob,present,2,sg,-,h,amxanag,ob,-,-,MV,active,39-1,39,*amxanagoba
2,amxanagob,present,2,sg,-,-,amxanag,ob,-,-,MV,active,39-1,39,*amxanagoba
3,hamxanagobs,present,3,sg,-,h,amxanag,ob,-,s,MV,active,39-1,39,*amxanagoba
4,amxanagobs,present,3,sg,-,-,amxanag,ob,-,s,MV,active,39-1,39,*amxanagoba


Display info about kv.

In [4]:
kv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294665 entries, 0 to 294664
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   form               294665 non-null  object
 1   tense_in_paradigm  294665 non-null  object
 2   person             294665 non-null  int64 
 3   number             294665 non-null  object
 4   preverb            294665 non-null  object
 5   pre2               294665 non-null  object
 6   root               294665 non-null  object
 7   sf2                294665 non-null  object
 8   caus_sf            294665 non-null  object
 9   ending             294659 non-null  object
 10  tsch_class         294665 non-null  object
 11  morph_type         294665 non-null  object
 12  sub_id             294665 non-null  object
 13  id                 294665 non-null  int64 
 14  vn                 294665 non-null  object
dtypes: int64(2), object(13)
memory usage: 33.7+ MB


Create a function to plot Missing data Ration in kv, print and save it in svg file.

In [5]:
def plot_nan(df: pd.DataFrame):
    if df.isnull().sum().sum() != 0:
        na_df = (df.isnull().sum() / len(df)) * 100
        print('How many elements are present in each files:')
        print(len(df)-df.isnull().sum())
        na_df = na_df.drop(na_df[na_df == 0].index).sort_values(ascending=False)
        missing_data = pd.DataFrame({'Missing Ratio %' :na_df})
        missing_data=round(missing_data,0)
        print(missing_data) 
        ax=missing_data.plot.barh(figsize=(10,3))
        ax.bar_label(ax.containers[0]) #rotation=270
        return ax
    else:
        print('No NAN found')

print('Kartu Verb Dataframe shape (rows,colomns) =',kv.shape)
#plot_nan(kv.replace('-',np.nan))

ax = plot_nan(kv.replace('-', np.nan))

now = datetime.now()
timestamp = now.strftime("%Y%m%d%H%M")
filename = f"output_missing_data_{timestamp}.svg"

ax.figure.tight_layout()
ax.figure.savefig(filename, format="svg")

plt.close(ax.figure)

Kartu Verb Dataframe shape (rows,colomns) = (294665, 15)
How many elements are present in each files:
form                 294665
tense_in_paradigm    294665
person               294665
number               294665
preverb              169378
pre2                 264752
root                 294665
sf2                  129237
caus_sf                4706
ending               287309
tsch_class           294665
morph_type           294665
sub_id               294665
id                   294665
vn                   294665
dtype: int64
         Missing Ratio %
caus_sf             98.0
sf2                 56.0
preverb             43.0
pre2                10.0
ending               2.0


Machine learning algorithms are not capable of directly processing textual data. In the case of the Kartu-Verbs database, which contains Georgian texts, it is necessary to convert the textual information into a numeric format. Additionally, it is important to handle empty values in the dataset, where empty values are denoted by a dash ("-").

To address these requirements, the following transformations were applied:
* Georgian strings were replaced with their equivalent binary representations.
* Empty values ("-") were replaced with 0.
* The 11 different values of "tense_in_paradigm" were replaced with a numerical range of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11].
* The 28 different values of "tsch_class" were replaced with a numerical range of [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27].
* The 5 different values of "morph_type" were replaced with a numerical range of [0, 1, 2, 3, 4].

Note: The values of "tsch_class" were substituted with the provided numerical range for simplicity and ease of representation.

original: IV1, IV2, IV3, IV4, KT, KT (nur mit i.O.), KT (OR), MV, P1, P2, P3, RM1, RM1 (OR), RM2, RM2 (OR), RM3, RM3 (OR), RM4, RM4(OR),RP1,RP1(mit,RP1(ohnei.O.),RP1(OR),RP2,RP2, (OR),RP3,RP3(OR),RP4,RP4(OR),RP5,RP5(OR),RP6,RP6, (OR),RP7,RP7(ohnei.O.),RP7(OR),T1,T1(OR),T2,T2(OR),T3, T3 (nur mit i.O.), T3 (OR), T4, T4 (nur mit i.O.), T4 (OR), T5, T5 (nur mit i.O.), T5 (OR), T5 (OR) (nur mit i.O.), ZP1, ZP2, ZP3

substitution: IV1, IV2, IV3, IV4, KT, MV, P1, P2, P3, RM1, RM2, RM3, RM4, RP1, RP2, RP3, RP4, RP5, RP6, RP7, T1, T2, T3, T4, T5, ZP1, ZP2, ZP3


In [6]:
kv['tense_in_paradigm'].replace(['present','imperfect','conj-present','future','conditional','conj-future','aorist','optative','perfect','pluperfect','conj-perfect'],[1,2,3,4,5,6,7,8,9,10,11], inplace=True)
kv['morph_type'].replace(['-','active','causative','passive','stative-passive'],[0,1,2,3,4], inplace=True)
kv['tsch_class'].replace(['IV1','IV2','IV3','IV4','KT','MV','P1','P2','P3','RM1','RM2','RM3','RM4','RP1','RP2','RP3','RP4','RP5','RP6','RP7','T1','T2','T3','T4','T5','ZP1','ZP2','ZP3'],[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28], inplace=True)
kv['number'].replace(['sg','pl'],[1,2],inplace=True)
kv['preverb'].replace(['-','a','amo','aR','aRmo','ga','gad','gada','gadmo','gamo','gan','gard','garda','garemo','garsSemo','garSemo','da','damo','Tana','iavar','mi','mimo','mo','uku','Se','Semo','STa','Ca','Camo','Zal','wa','wamo','war','warmo','wina'],[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34], inplace=True)
kv['pre2'].replace(['-'],['0'],inplace=True)
kv['root'].replace(['-'],['0'],inplace=True)
kv['sf2'].replace(['-','av','am','e','eb','ev','v','i','m','ob','of'],[0,1,2,3,4,5,6,7,8,9,10],inplace=True)
kv['caus_sf'].replace(['-','evin','in'],[0,1,2],inplace=True)
kv['ending'].replace(['-'],['0'],inplace=True)
kv['sub_id'].replace('.*-','',regex=True,inplace=True)
kv['vn'].replace(['-'],['0'],inplace=True)

In [7]:
kv[['sub_id']] = kv[['sub_id']].apply(pd.to_numeric) #to convert sub_id object type to int64
#print(kv['sub_id'])

Create a function to sum up ascii character decimal values

In [8]:
def str2dec_ascii(x):
    if pd.isna(x):
        return 0
    x = str(x)
    return sum(ord(ch) for ch in x)

#kv[kv['ending'].isna()][['ending']].head(10)

Subsequently, we incorporated additional fields into the "kv" dataframe, specifically "formd", "preverbd", "pre2d", "rootd", "caus_sfd", "sf2d", and "endingd". These newly introduced fields correspond to the decimal representations of the original fields, namely "form", "preverb", "pre2", "root", "caus_sf", "sf2", and "ending". The purpose of including these fields is to store the converted decimal representations of the respective values.

In [9]:
#To convert a tranliterated Georgian string into the sum of its characters' decimal representations,
kv['formd'] = kv['form'].apply(lambda x: str2dec_ascii(x))
kv['pre2d'] = kv['pre2'].apply(lambda x: str2dec_ascii(x))
kv['rootd'] = kv['root'].apply(lambda x: str2dec_ascii(x))
kv['endingd'] = kv['ending'].apply(lambda x: str2dec_ascii(x))
kv['vnd'] = kv['vn'].apply(lambda x: str2dec_ascii(x))

To encode the field "vn2d" by enumerating its values from 1 to N, and store this encoding information in a dictionary called "index_vn2."

In [10]:
index_vn = {}
vn_new_list = []

for i in kv['vn']:
    if i not in index_vn:
        index_vn[i] = len(index_vn)
    vn_new_list.append(index_vn[i])

kv['vnd']=vn_new_list

#print(index_vn)
filename = f"output_vn_index_{timestamp}.txt"
with open(filename,'w') as data:
    data.write(str(index_vn))

To facilitate further investigation, we can save the corresponding text and numeric representations for the "forms" and "verbal nouns" separately in individual files. This separation will allow for easier analysis and examination of the data.

In [11]:
filename = f"output_form_formd_{timestamp}.txt"
kv[['form','formd']].to_csv(filename,sep=',')
filename = f"output_vn_vnd_{timestamp}.txt"
kv[['vn','vnd']].to_csv(filename,sep=',')

In [12]:
kv.describe()

,tense_in_paradigm,person,number,preverb,sf2,caus_sf,tsch_class,morph_type,sub_id,id,formd,pre2d,rootd,endingd,vnd
count,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000
mean,6.248808,2.112463,1.531186,7.731376,1.894341,0.018000,14.402491,1.814467,25.888063,2059.900219,1080.482568,147.942538,363.915463,353.269845,250.425222
std,3.137606,0.813681,0.499027,9.993178,2.502208,0.147428,8.049219,0.893504,27.425140,1214.070507,314.873097,77.038147,144.046584,202.951519,137.672942
min,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,39.000000,195.000000,48.000000,82.000000,0.000000,0.000000
25%,3.000000,1.000000,1.000000,0.000000,0.000000,0.000000,6.000000,1.000000,7.000000,904.000000,847.000000,101.000000,305.000000,202.000000,140.000000
50%,7.000000,2.000000,2.000000,1.000000,0.000000,0.000000,14.000000,1.000000,16.000000,2154.000000,1056.000000,115.000000,336.000000,308.000000,239.000000
75%,9.000000,3.000000,2.000000,16.000000,4.000000,0.000000,22.000000,3.000000,38.000000,3218.000000,1280.000000,214.000000,439.000000,507.000000,368.000000
max,11.000000,3.000000,2.000000,33.000000,9.000000,2.000000,28.000000,4.000000,222.000000,3844.000000,2642.000000,762.000000,956.000000,1207.000000,493.000000


In [13]:
kv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294665 entries, 0 to 294664
Data columns (total 20 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   form               294665 non-null  object
 1   tense_in_paradigm  294665 non-null  int64 
 2   person             294665 non-null  int64 
 3   number             294665 non-null  int64 
 4   preverb            294665 non-null  int64 
 5   pre2               294665 non-null  object
 6   root               294665 non-null  object
 7   sf2                294665 non-null  int64 
 8   caus_sf            294665 non-null  int64 
 9   ending             294659 non-null  object
 10  tsch_class         294665 non-null  int64 
 11  morph_type         294665 non-null  int64 
 12  sub_id             294665 non-null  int64 
 13  id                 294665 non-null  int64 
 14  vn                 294665 non-null  object
 15  formd              294665 non-null  int64 
 16  pre2d              2

# Setting up the Data
* Create a new dataframe called "kn_n" where we will retain only the number representation of the data.
* Simplify the dataframe by renaming the "tense_in_paradigm" column to "tense" for the sake of simplicity and clarity.

In [14]:
kv_n = kv.loc[:,['formd','tense_in_paradigm','person','number','preverb','pre2d','rootd','sf2','caus_sf','endingd','tsch_class','morph_type','sub_id','id','vnd']]
kv_n.rename(columns={'tense_in_paradigm':'tense'}, inplace=True) # just rename 'tense_in_paradigm' with 'tense' for simplicity

To free up Memory, you can delete the "kv" dataframe.

In [15]:
del kv

In [16]:
kv_n.describe()

,formd,tense,person,number,preverb,pre2d,rootd,sf2,caus_sf,endingd,tsch_class,morph_type,sub_id,id,vnd
count,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000,294665.000000
mean,1080.482568,6.248808,2.112463,1.531186,7.731376,147.942538,363.915463,1.894341,0.018000,353.269845,14.402491,1.814467,25.888063,2059.900219,250.425222
std,314.873097,3.137606,0.813681,0.499027,9.993178,77.038147,144.046584,2.502208,0.147428,202.951519,8.049219,0.893504,27.425140,1214.070507,137.672942
min,195.000000,1.000000,1.000000,1.000000,0.000000,48.000000,82.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,39.000000,0.000000
25%,847.000000,3.000000,1.000000,1.000000,0.000000,101.000000,305.000000,0.000000,0.000000,202.000000,6.000000,1.000000,7.000000,904.000000,140.000000
50%,1056.000000,7.000000,2.000000,2.000000,1.000000,115.000000,336.000000,0.000000,0.000000,308.000000,14.000000,1.000000,16.000000,2154.000000,239.000000
75%,1280.000000,9.000000,3.000000,2.000000,16.000000,214.000000,439.000000,4.000000,0.000000,507.000000,22.000000,3.000000,38.000000,3218.000000,368.000000
max,2642.000000,11.000000,3.000000,2.000000,33.000000,762.000000,956.000000,9.000000,2.000000,1207.000000,28.000000,4.000000,222.000000,3844.000000,493.000000


In [17]:
kv_n.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294665 entries, 0 to 294664
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   formd       294665 non-null  int64
 1   tense       294665 non-null  int64
 2   person      294665 non-null  int64
 3   number      294665 non-null  int64
 4   preverb     294665 non-null  int64
 5   pre2d       294665 non-null  int64
 6   rootd       294665 non-null  int64
 7   sf2         294665 non-null  int64
 8   caus_sf     294665 non-null  int64
 9   endingd     294665 non-null  int64
 10  tsch_class  294665 non-null  int64
 11  morph_type  294665 non-null  int64
 12  sub_id      294665 non-null  int64
 13  id          294665 non-null  int64
 14  vnd         294665 non-null  int64
dtypes: int64(15)
memory usage: 33.7 MB


To reduce the memory usage of a variable, we consider changing its data type to a less memory-intensive alternative.

In [18]:
print(kv_n.dtypes)
print(kv_n['tense'].dtypes)

formd         int64
tense         int64
person        int64
number        int64
preverb       int64
pre2d         int64
rootd         int64
sf2           int64
caus_sf       int64
endingd       int64
tsch_class    int64
morph_type    int64
sub_id        int64
id            int64
vnd           int64
dtype: object
int64


In [19]:
kv_n[kv_n.columns[0]]=kv_n.loc[:,'formd'].astype('int16')
kv_n[kv_n.columns[1]]=kv_n.loc[:,'tense'].astype('byte')
kv_n[kv_n.columns[2]]=kv_n.loc[:,'person'].astype('byte')
kv_n[kv_n.columns[3]]=kv_n.loc[:,'number'].astype('byte')
kv_n[kv_n.columns[4]]=kv_n.loc[:,'preverb'].astype('byte')
kv_n[kv_n.columns[5]]=kv_n.loc[:,'pre2d'].astype('int16')
kv_n[kv_n.columns[6]]=kv_n.loc[:,'rootd'].astype('int16')
kv_n[kv_n.columns[7]]=kv_n.loc[:,'sf2'].astype('byte')
kv_n[kv_n.columns[8]]=kv_n.loc[:,'caus_sf'].astype('byte')
kv_n[kv_n.columns[9]]=kv_n.loc[:,'endingd'].astype('int16')
kv_n[kv_n.columns[10]]=kv_n.loc[:,'tsch_class'].astype('byte')
kv_n[kv_n.columns[11]]=kv_n.loc[:,'morph_type'].astype('byte')
kv_n[kv_n.columns[12]]=kv_n.loc[:,'id'].astype('int16')
kv_n[kv_n.columns[13]]=kv_n.loc[:,'sub_id'].astype('int16') #float16
kv_n[kv_n.columns[14]]=kv_n.loc[:,'vnd'].astype('int16')

We can examine the new dataframe.

In [20]:
kv_n.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294665 entries, 0 to 294664
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   formd       294665 non-null  int16
 1   tense       294665 non-null  int8 
 2   person      294665 non-null  int8 
 3   number      294665 non-null  int8 
 4   preverb     294665 non-null  int8 
 5   pre2d       294665 non-null  int16
 6   rootd       294665 non-null  int16
 7   sf2         294665 non-null  int8 
 8   caus_sf     294665 non-null  int8 
 9   endingd     294665 non-null  int16
 10  tsch_class  294665 non-null  int8 
 11  morph_type  294665 non-null  int8 
 12  sub_id      294665 non-null  int16
 13  id          294665 non-null  int16
 14  vnd         294665 non-null  int16
dtypes: int16(7), int8(8)
memory usage: 6.2 MB


In [21]:
print(kv_n)

#filename = f"output_vnd_{timestamp}.csv"
#kv_n.to_csv(filename, index=False, encoding="utf-8")

        formd  tense  person  number  preverb  pre2d  rootd  sf2  caus_sf  \
0        1060      1       1       1        0    118    733    9        0   
1        1046      1       2       1        0    104    733    9        0   
2         942      1       2       1        0     48    733    9        0   
3        1161      1       3       1        0    104    733    9        0   
4        1057      1       3       1        0     48    733    9        0   
...       ...    ...     ...     ...      ...    ...    ...  ...      ...   
294660    772      7       2       1        0    453    319    0        0   
294661    629      1       1       1       24    223    222    0        0   
294662    511      1       2       1       24    105    222    0        0   
294663    629      4       1       1       24    223    222    0        0   
294664    511      4       2       1       24    105    222    0        0   

        endingd  tsch_class  morph_type  sub_id    id  vnd  
0            4

## Decision Tree Model - Solution

## Train Test Split

Now, we will proceed with the task of dividing our data into a training set and a testing set.

To accomplish this, we will use the functionality provided by the scikit-learn library. This allows us to easily split our data into two distinct sets: one for training our model and the other for evaluating its performance.

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
X = kv_n.drop('vnd',axis=1)
y = kv_n['vnd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)
#print(X_test)
#print(y)
#X_test.to_csv(r'X_test.txt', index=None, sep=';')

Import DecisionTreeClassifier

In [24]:
from sklearn.tree import DecisionTreeClassifier

Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.

In [25]:
dtree = DecisionTreeClassifier()  #(criterion="log_loss", splitter="random", max_depth=16);
dtree.fit(X_train,y_train)

DecisionTreeClassifier()

## Predictions and Evaluation of Decision Tree
Create predictions from the test set and create a classification report and a confusion matrix.

In [26]:
predictions = dtree.predict(X_test)

In [27]:
from sklearn.metrics import classification_report,confusion_matrix
print(classification_report(y_test,predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        37
           1       1.00      1.00      1.00       207
           2       1.00      1.00      1.00       136
           3       1.00      1.00      1.00        47
           4       1.00      1.00      1.00       180
           5       1.00      1.00      1.00        37
           6       1.00      1.00      1.00        42
           7       1.00      1.00      1.00        51
           8       1.00      1.00      1.00        68
           9       1.00      1.00      1.00        35
          10       1.00      1.00      1.00        24
          11       1.00      1.00      1.00       136
          12       1.00      1.00      1.00        37
          13       1.00      1.00      1.00       159
          14       1.00      1.00      1.00        30
          15       1.00      1.00      1.00        51
          16       1.00      1.00      1.00        34
          17       1.00    

To store the clssificaion report.

In [28]:
report = classification_report(y_test,predictions)
report_path = "report.txt"
filename = f"output_report_{timestamp}.txt"
text_file = open(filename,"w")
n = text_file.write(report)
text_file.close()

In [29]:
print(confusion_matrix(y_test,predictions))
cm = confusion_matrix(y_test,predictions)

[[ 37   0   0 ...   0   0   0]
 [  0 207   0 ...   0   0   0]
 [  0   0 136 ...   0   0   0]
 ...
 [  0   0   0 ...   8   0   0]
 [  0   0   0 ...   0   3   0]
 [  0   0   0 ...   0   0  10]]


# Prepare test data from a file

To prepare the test data from a file, we have the "data_vn-.csv" file that includes all the fields except for the Verbal Noun. It is particularly intriguing to observe how our trained model performs in predicting the missing Verbal Nouns.

The file "data_vn-.csv" consists of 599813 lines and will serve as our test dataset for evaluating the model's ability to predict the missing Verbal Nouns.

## Get the Data
Read in the "data_vn-.csv" file and set it to a data frame called kv.

In [30]:
X_test2 = pd.read_csv('data_vn-lat', sep=';')

Check the head of the kv dataframe.

In [31]:
#print(X_test2)
Solution = X_test2.copy()

In [32]:
X_test2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599812 entries, 0 to 599811
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   form               599812 non-null  object
 1   tense_in_paradigm  599812 non-null  object
 2   person             599812 non-null  int64 
 3   number             599812 non-null  object
 4   preverb            599812 non-null  object
 5   pre2               599812 non-null  object
 6   root               599812 non-null  object
 7   sf2                599812 non-null  object
 8   caus_sf            599812 non-null  object
 9   ending             599808 non-null  object
 10  tsch_class         599812 non-null  object
 11  morph_type         599812 non-null  object
 12  sub_id             599812 non-null  object
 13  id                 599812 non-null  int64 
 14  vn                 599812 non-null  object
dtypes: int64(2), object(13)
memory usage: 68.6+ MB


To prepare the data, we will proceed with converting the textual information into a numeric representation. This conversion is necessary to enable the utilization of machine learning algorithms that operate on numerical data.

In [33]:
X_test2['tense_in_paradigm'].replace(['present','imperfect','conj-present','future','conditional','conj-future','aorist','optative','perfect','pluperfect','conj-perfect'],[1,2,3,4,5,6,7,8,9,10,11], inplace=True)
X_test2['morph_type'].replace(['-','active','causative','passive','stative-passive'],[0,1,2,3,4], inplace=True)
X_test2['tsch_class'].replace(['IV1','IV2','IV3','IV4','KT','MV','P1','P2','P3','RM1','RM2','RM3','RM4','RP1','RP2','RP3','RP4','RP5','RP6','RP7','T1','T2','T3','T4','T5','ZP1','ZP2','ZP3'],[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28], inplace=True)
X_test2['number'].replace(['sg','pl'],[1,2],inplace=True)
X_test2['preverb'].replace(['-','a','amo','aR','aRmo','ga','gad','gada','gadmo','gamo','gan','gard','garda','garemo','garsSemo','garSemo','da','damo','Tana','iavar','mi','mimo','mo','uku','Se','Semo','STa','Ca','Camo','Zal','wa','wamo','war','warmo','wina','gardmo','zewamo','iZulebul','naTel','srul','uar','ugulebel','uvnebel','uzrunvel','ukvdav','uCinar','RaRad','Seuracx','cxad','xel','winaaR'],[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50], inplace=True)
X_test2['pre2'].replace(['-'],['0'],inplace=True)
X_test2['root'].replace(['-'],['0'],inplace=True)
X_test2['sf2'].replace(['-','av','am','e','eb','ev','v','i','m','ob','of'],[0,1,2,3,4,5,6,7,8,9,10],inplace=True)
X_test2['caus_sf'].replace(['-','evin','in'],[0,1,2],inplace=True)
X_test2['ending'].replace(['-'],['0'],inplace=True)
X_test2['sub_id'].replace('.*-','',regex=True,inplace=True)

X_test2[['sub_id']] = X_test2[['sub_id']].apply(pd.to_numeric)

In [34]:
X_test2['formd'] = X_test2['form'].apply(lambda x: str2dec_ascii(x))
X_test2['pre2d'] = X_test2['pre2'].apply(lambda x: str2dec_ascii(x))
X_test2['rootd'] = X_test2['root'].apply(lambda x: str2dec_ascii(x))
X_test2['endingd'] = X_test2['ending'].apply(lambda x: str2dec_ascii(x))

In [35]:
X_test2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599812 entries, 0 to 599811
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   form               599812 non-null  object
 1   tense_in_paradigm  599812 non-null  int64 
 2   person             599812 non-null  int64 
 3   number             599812 non-null  int64 
 4   preverb            599812 non-null  int64 
 5   pre2               599812 non-null  object
 6   root               599812 non-null  object
 7   sf2                599812 non-null  int64 
 8   caus_sf            599812 non-null  int64 
 9   ending             599808 non-null  object
 10  tsch_class         599812 non-null  int64 
 11  morph_type         599812 non-null  int64 
 12  sub_id             599812 non-null  int64 
 13  id                 599812 non-null  int64 
 14  vn                 599812 non-null  object
 15  formd              599812 non-null  int64 
 16  pre2d              5

In [36]:
X_test2 = X_test2.loc[:,['formd','tense_in_paradigm','person','number','preverb','pre2d','rootd','sf2','caus_sf','endingd','tsch_class','morph_type','sub_id','id']]
X_test2.rename(columns={'tense_in_paradigm':'tense'}, inplace=True) # just rename 'tense_in_paradigm' with 'tense' for simplicity

In [37]:
X_test2[X_test2.columns[0]]=X_test2.loc[:,'formd'].astype('int16')
X_test2[X_test2.columns[1]]=X_test2.loc[:,'tense'].astype('byte')
X_test2[X_test2.columns[2]]=X_test2.loc[:,'person'].astype('byte')
X_test2[X_test2.columns[3]]=X_test2.loc[:,'number'].astype('byte')
X_test2[X_test2.columns[4]]=X_test2.loc[:,'preverb'].astype('byte')
X_test2[X_test2.columns[5]]=X_test2.loc[:,'pre2d'].astype('int16')
X_test2[X_test2.columns[6]]=X_test2.loc[:,'rootd'].astype('int16')
X_test2[X_test2.columns[7]]=X_test2.loc[:,'sf2'].astype('byte')
X_test2[X_test2.columns[8]]=X_test2.loc[:,'caus_sf'].astype('byte')
X_test2[X_test2.columns[9]]=X_test2.loc[:,'endingd'].astype('int16')
X_test2[X_test2.columns[10]]=X_test2.loc[:,'tsch_class'].astype('byte')
X_test2[X_test2.columns[11]]=X_test2.loc[:,'morph_type'].astype('byte')
X_test2[X_test2.columns[12]]=X_test2.loc[:,'id'].astype('int16')
X_test2[X_test2.columns[13]]=X_test2.loc[:,'sub_id'].astype('int16') #float16

In [38]:
X_test2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 599812 entries, 0 to 599811
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   formd       599812 non-null  int16
 1   tense       599812 non-null  int8 
 2   person      599812 non-null  int8 
 3   number      599812 non-null  int8 
 4   preverb     599812 non-null  int8 
 5   pre2d       599812 non-null  int16
 6   rootd       599812 non-null  int16
 7   sf2         599812 non-null  int8 
 8   caus_sf     599812 non-null  int8 
 9   endingd     599812 non-null  int16
 10  tsch_class  599812 non-null  int8 
 11  morph_type  599812 non-null  int8 
 12  sub_id      599812 non-null  int16
 13  id          599812 non-null  int16
dtypes: int16(6), int8(8)
memory usage: 11.4 MB


In [39]:
#print(X_test2)

## Predictions and Evaluation of Decision Tree
Generate predictions from the test set and then create a classification report and a confusion matrix to evaluate the performance of the model.

In [40]:
predictions2 = dtree.predict(X_test2)

In [41]:
print(predictions2)

filename = f"output_predictions2_{timestamp}.txt"

with open(filename, 'w') as f:
    for line in predictions2:
        f.write(f"{line}\n")

[  1   1   1 ... 475 475 475]


After generating predictions from the test set, we will proceed to decode the numeric predictions back into their original Georgian text representations. This step allows us to interpret and analyze the model's outputs in a more understandable and meaningful manner. By converting the numeric predictions back to Georgian text, we can gain insights into the predicted outcomes and assess the model's performance in a linguistically meaningful context.

In [42]:
tmppd = pd.DataFrame({'vn': predictions2})
#print(tmppd)
#To reverse index_vn2 - swap keys:values.
index_vn_rev = {i: j for j, i in index_vn.items()}
#print(index_vn)
Solution['vn'] = tmppd.replace(index_vn_rev)
#print(Solution)

In [43]:
filename = f"output_solution{timestamp}.csv"
Solution.to_csv(filename, index=False, sep=';')

# Conclusion

The decision tree model has demonstrated excellent predictive capabilities. We conducted multiple runs of the model on the same dataset, each time using a different random_state parameter for the training and testing data split. Remarkably, in all runs, the model consistently produced identical results.

The model provides predictions with 98-99% as detailed in the **output_report.txt** file. These probabilities offer insights into the model's confidence levels for each prediction.

The corresponding outcomes can be accessed in the **output_solution.csv** file, which presents the predicted results derived from the decision tree model.